# Language Translation Tool
A web-based translation tool built with Google Cloud Translation API and Gradio UI.
Supports 40+ languages with text-to-speech and clipboard copy functionality.

In [30]:
# installing required libraries
pip install google-cloud-translate gradio gTTS

Note: you may need to restart the kernel to use updated packages.


In [15]:
# testing the api functionality
import requests
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GOOGLE_TRANSLATE_API_KEY")

url = "https://translation.googleapis.com/language/translate/v2"

params = {
    "q": "Hello, how are you?",
    "target": "fr",
    "key": API_KEY
}

response = requests.post(url, params=params)
result = response.json()
print(result["data"]["translations"][0]["translatedText"])

Bonjour comment allez-vous?


In [11]:
# importing and verifying gradio version
import gradio as gr
print(gr.__version__)

6.14.0


In [31]:
# building the main translation tool
import requests
from dotenv import load_dotenv
import os
import gradio as gr
from gtts import gTTS
import tempfile

load_dotenv()
API_KEY = os.getenv("GOOGLE_TRANSLATE_API_KEY")

# Language options
LANGUAGES = {
    "Afrikaans": "af", "Arabic": "ar", "Bengali": "bn", "Chinese (Simplified)": "zh-CN",
    "Chinese (Traditional)": "zh-TW", "Czech": "cs", "Danish": "da", "Dutch": "nl",
    "English": "en", "Finnish": "fi", "French": "fr", "German": "de",
    "Greek": "el", "Gujarati": "gu", "Hebrew": "iw", "Hindi": "hi",
    "Hungarian": "hu", "Indonesian": "id", "Italian": "it", "Japanese": "ja",
    "Kannada": "kn", "Korean": "ko", "Malay": "ms", "Malayalam": "ml",
    "Marathi": "mr", "Norwegian": "no", "Persian": "fa", "Polish": "pl",
    "Portuguese": "pt", "Punjabi": "pa", "Romanian": "ro", "Russian": "ru",
    "Spanish": "es", "Swahili": "sw", "Swedish": "sv", "Tamil": "ta",
    "Telugu": "te", "Thai": "th", "Turkish": "tr", "Ukrainian": "uk",
    "Urdu": "ur", "Vietnamese": "vi"
}

LANGUAGE_NAMES = list(LANGUAGES.keys())

# Translation function
def translate_text(text, source_lang, target_lang):
    if not text.strip():
        return "Error! Please enter some text to translate.", None
    if source_lang == target_lang:
        return "Error! Source and target languages must be different.", None
    source_code = LANGUAGES[source_lang]
    target_code = LANGUAGES[target_lang]
    url = "https://translation.googleapis.com/language/translate/v2"
    params = {
        "q": text,
        "source": source_code,
        "target": target_code,
        "key": API_KEY
    }
    response = requests.post(url, params=params)
    result = response.json()
    if "data" in result:
        translated = result["data"]["translations"][0]["translatedText"]
        return translated, None
    else:
        error = result.get("error", {}).get("message", "Unknown error")
        return f"Error: {error}", None

# Text-to-speech function
def speak_translation(translated_text, target_lang):
    if not translated_text.strip():
        return None
    if translated_text.startswith("Error!") or translated_text.startswith("Error"):
        return None
    target_code = LANGUAGES[target_lang]
    try:
        tts = gTTS(text=translated_text, lang=target_code)
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        tts.save(tmp.name)
        return tmp.name
    except Exception as e:
        return None

# Copy function
def copy_text(text):
    if not text.strip():
        return "Error! Nothing to copy."
    if text.startswith("Error! ") or text.startswith("Error"):
        return "Cannot copy! No valid translation."
    return "Copied to clipboard!"

# Gradio UI 
with gr.Blocks(title="Language Translation Tool") as app:
    gr.Markdown("# Language Translation Tool")
    gr.Markdown("Translate text between 40+ languages with text-to-speech support.")

    with gr.Row():
        source_lang = gr.Dropdown(
            choices=LANGUAGE_NAMES,
            value="English",
            label="Source Language"
        )
        target_lang = gr.Dropdown(
            choices=LANGUAGE_NAMES,
            value="French",
            label="Target Language"
        )

    input_text = gr.Textbox(
        lines=4,
        placeholder="Enter text to translate...",
        label="Input Text"
    )

    translate_btn = gr.Button("Translate", variant="primary")

    output_text = gr.Textbox(
        lines=4,
        label="Translated Text"
    )

    copy_btn = gr.Button("Copy Translation", variant="secondary")
    copy_status = gr.Textbox(label="", interactive=False, visible=True, container=False)

    tts_btn = gr.Button("Speak Translation", variant="secondary")
    audio_output = gr.Audio(label="Text-to-Speech Output", type="filepath")

# Button actions
    translate_btn.click(
        fn=translate_text,
        inputs=[input_text, source_lang, target_lang],
        outputs=[output_text, audio_output]
    )

    copy_btn.click(
        fn=copy_text,
        inputs=[output_text],
        outputs=[copy_status],
        js="(text) => { navigator.clipboard.writeText(text); return text; }"
    )

    tts_btn.click(
        fn=speak_translation,
        inputs=[output_text, target_lang],
        outputs=audio_output
    )

app.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
